# Data Cleaning

This notebook aims to explore and understand the Spanish electricity consumption dataset's structure and clean it by removing duplicates, fixing schema datatypes, and finally checking null values. The cleaned data is at last exported as a parquet file.

## Setup

We use Polars because of its lazy loading feature. Lazy Parquet scanning allows us to work with the data without loading all 16 million records into memory at once.

In [1]:
from pathlib import Path
import polars as pl

In [2]:
# Project root
PROJECT_ROOT = Path.cwd().parent

A_PATH = PROJECT_ROOT / "data" / "a.parquet"

if not A_PATH.exists():
    raise FileNotFoundError(
        f"Could not find Table A at: {A_PATH}\n"
        "Make sure the notebook is inside the EDA folder."
    )

# Keep Table A lazy throughout the main analysis.
A_raw = pl.scan_parquet(str(A_PATH))

print("Table A found:", A_PATH)
print("Table A is being read lazily.")

Table A found: e:\tasks\spain_elec\data\a.parquet
Table A is being read lazily.


## Raw dataset

In [3]:
raw_schema = A_raw.collect_schema()

A_rows = (
    A_raw
    .select(pl.len())
    .collect()
    .item()
)

print(f"Rows: {A_rows:,}")
print(f"Columns: {len(raw_schema.names())}")

Rows: 16,155,725
Columns: 83


In [4]:
A_raw.head(5).collect()

DIA,H1,ACTIVA_H1,REACTIVA_H1,H2,ACTIVA_H2,REACTIVA_H2,H3,ACTIVA_H3,REACTIVA_H3,H4,ACTIVA_H4,REACTIVA_H4,H5,ACTIVA_H5,REACTIVA_H5,H6,ACTIVA_H6,REACTIVA_H6,H7,ACTIVA_H7,REACTIVA_H7,H8,ACTIVA_H8,REACTIVA_H8,H9,ACTIVA_H9,REACTIVA_H9,H10,ACTIVA_H10,REACTIVA_H10,H11,ACTIVA_H11,REACTIVA_H11,H12,ACTIVA_H12,REACTIVA_H12,…,H16,ACTIVA_H16,REACTIVA_H16,H17,ACTIVA_H17,REACTIVA_H17,H18,ACTIVA_H18,REACTIVA_H18,H19,ACTIVA_H19,REACTIVA_H19,H20,ACTIVA_H20,REACTIVA_H20,H21,ACTIVA_H21,REACTIVA_H21,H22,ACTIVA_H22,REACTIVA_H22,H23,ACTIVA_H23,REACTIVA_H23,H24,ACTIVA_H24,REACTIVA_H24,H25,ACTIVA_H25,REACTIVA_H25,DE_MUNICIP,FECHA_ALTA_STRO,TARGET_TENENCIA_CUPS,IDENTIFICADOR,CNAE,PRODUCTO,MERCADO
str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,str,str,str,i64,str,str,str
"""20150624""",1,280,0,2,228,0,3,172,0,4,196,0,5,287,0,6,195,0,7,223,0,8,254,0,9,274,0,10,404,0,11,333,0,12,1106,0,…,16,262,0,17,300,0,18,326,0,19,281,0,20,357,0,21,448,0,22,349,0,23,501,0,24,383,0,25,0,0,"""UBRIQUE""","""Q11989""","""0.127242""",26785,"""T1""","""P1""","""M1"""
"""20150913""",1,142,0,2,61,0,3,67,0,4,64,0,5,64,0,6,70,0,7,126,0,8,101,0,9,40,0,10,179,0,11,131,0,12,169,0,…,16,90,0,17,50,0,18,87,0,19,48,0,20,130,0,21,366,0,22,302,0,23,133,0,24,81,0,25,0,0,"""SANTA COLOMA DE GRAMENET""","""Q31979""","""0.000000""",86782,"""T1""","""P13""","""M2"""
"""20150710""",1,0,0,2,0,0,3,0,0,4,0,0,5,0,0,6,0,0,7,0,0,8,0,0,9,0,0,10,0,0,11,0,0,12,0,0,…,16,0,0,17,0,0,18,0,0,19,0,0,20,0,0,21,0,0,22,0,0,23,0,0,24,0,0,25,0,0,"""VALVERDE""","""Q12010""","""0.000000""",75439,"""T1""","""P14""","""M2"""
"""20150517""",1,186,0,2,129,0,3,176,0,4,123,0,5,183,0,6,200,0,7,185,0,8,170,0,9,161,0,10,409,0,11,399,0,12,351,0,…,16,422,0,17,321,0,18,330,0,19,346,0,20,329,0,21,348,0,22,345,0,23,142,0,24,236,0,25,0,0,"""CAZORLA""","""Q31989""","""0.000000""",43916,"""T1""","""P1""","""M1"""
"""20141108""",1,48,0,2,86,0,3,115,0,4,50,0,5,48,0,6,108,0,7,119,0,8,59,0,9,59,0,10,106,0,11,113,0,12,59,0,…,16,112,0,17,65,0,18,681,0,19,731,0,20,493,0,21,466,0,22,231,0,23,67,0,24,67,0,25,0,0,"""HUELVA""","""Q31989""","""0.011962""",18260,"""T1""","""P1""","""M1"""


## New column names

In [5]:
column_mapping = {
    "DIA": "DATE",
    "DE_MUNICIP": "MUNICIPALITY",
    "FECHA_ALTA_STRO": "SUPPLY_REGISTRATION_PERIOD",
    "TARGET_TENENCIA_CUPS": "GAS_DISTRIBUTION_PROBABILITY",
    "IDENTIFICADOR": "CUSTOMER_ID",
    "CNAE": "CUSTOMER_TYPE",
    "PRODUCTO": "ELECTRICITY_PRODUCT",
    "MERCADO": "MARKET",
}

for i in range(1, 26):
    column_mapping[f"H{i}"] = f"TIME_H{i}"
    column_mapping[f"ACTIVA_H{i}"] = f"ACTIVE_H{i}"
    column_mapping[f"REACTIVA_H{i}"] = f"REACTIVE_H{i}"

A = A_raw.rename(column_mapping)

print("Columns after renaming:")
print(A.collect_schema().names())

Columns after renaming:
['DATE', 'TIME_H1', 'ACTIVE_H1', 'REACTIVE_H1', 'TIME_H2', 'ACTIVE_H2', 'REACTIVE_H2', 'TIME_H3', 'ACTIVE_H3', 'REACTIVE_H3', 'TIME_H4', 'ACTIVE_H4', 'REACTIVE_H4', 'TIME_H5', 'ACTIVE_H5', 'REACTIVE_H5', 'TIME_H6', 'ACTIVE_H6', 'REACTIVE_H6', 'TIME_H7', 'ACTIVE_H7', 'REACTIVE_H7', 'TIME_H8', 'ACTIVE_H8', 'REACTIVE_H8', 'TIME_H9', 'ACTIVE_H9', 'REACTIVE_H9', 'TIME_H10', 'ACTIVE_H10', 'REACTIVE_H10', 'TIME_H11', 'ACTIVE_H11', 'REACTIVE_H11', 'TIME_H12', 'ACTIVE_H12', 'REACTIVE_H12', 'TIME_H13', 'ACTIVE_H13', 'REACTIVE_H13', 'TIME_H14', 'ACTIVE_H14', 'REACTIVE_H14', 'TIME_H15', 'ACTIVE_H15', 'REACTIVE_H15', 'TIME_H16', 'ACTIVE_H16', 'REACTIVE_H16', 'TIME_H17', 'ACTIVE_H17', 'REACTIVE_H17', 'TIME_H18', 'ACTIVE_H18', 'REACTIVE_H18', 'TIME_H19', 'ACTIVE_H19', 'REACTIVE_H19', 'TIME_H20', 'ACTIVE_H20', 'REACTIVE_H20', 'TIME_H21', 'ACTIVE_H21', 'REACTIVE_H21', 'TIME_H22', 'ACTIVE_H22', 'REACTIVE_H22', 'TIME_H23', 'ACTIVE_H23', 'REACTIVE_H23', 'TIME_H24', 'ACTIVE_H24', 'R

## Schema Datatypes Fix

In [6]:
schema_table = pl.DataFrame({
    "column": A.collect_schema().names(),
    "dtype": [str(dtype) for dtype in raw_schema.dtypes()],
})

for col, dtype in zip(schema_table["column"], schema_table["dtype"]):
    print(f"{col}: {dtype}")

DATE: String
TIME_H1: Int32
ACTIVE_H1: Int32
REACTIVE_H1: Int32
TIME_H2: Int32
ACTIVE_H2: Int32
REACTIVE_H2: Int32
TIME_H3: Int32
ACTIVE_H3: Int32
REACTIVE_H3: Int32
TIME_H4: Int32
ACTIVE_H4: Int32
REACTIVE_H4: Int32
TIME_H5: Int32
ACTIVE_H5: Int32
REACTIVE_H5: Int32
TIME_H6: Int32
ACTIVE_H6: Int32
REACTIVE_H6: Int32
TIME_H7: Int32
ACTIVE_H7: Int32
REACTIVE_H7: Int32
TIME_H8: Int32
ACTIVE_H8: Int32
REACTIVE_H8: Int32
TIME_H9: Int32
ACTIVE_H9: Int32
REACTIVE_H9: Int32
TIME_H10: Int32
ACTIVE_H10: Int32
REACTIVE_H10: Int32
TIME_H11: Int32
ACTIVE_H11: Int32
REACTIVE_H11: Int32
TIME_H12: Int32
ACTIVE_H12: Int32
REACTIVE_H12: Int32
TIME_H13: Int32
ACTIVE_H13: Int32
REACTIVE_H13: Int32
TIME_H14: Int32
ACTIVE_H14: Int32
REACTIVE_H14: Int32
TIME_H15: Int32
ACTIVE_H15: Int32
REACTIVE_H15: Int32
TIME_H16: Int32
ACTIVE_H16: Int32
REACTIVE_H16: Int32
TIME_H17: Int32
ACTIVE_H17: Int32
REACTIVE_H17: Int32
TIME_H18: Int32
ACTIVE_H18: Int32
REACTIVE_H18: Int32
TIME_H19: Int32
ACTIVE_H19: Int32
REACTIVE

In [7]:
# checking number of empty strings "" in string columns
# numbers should match null counts after column conversion

string_cols = [col for col, dt in list(zip(schema_table["column"], schema_table["dtype"])) if dt == 'String']

empty_counts = (
    A
    .select([
        (pl.col(c) == "").sum().alias(c)
        for c in string_cols
    ])
    .collect()
)

empty_counts

DATE,MUNICIPALITY,SUPPLY_REGISTRATION_PERIOD,GAS_DISTRIBUTION_PROBABILITY,CUSTOMER_TYPE,ELECTRICITY_PRODUCT,MARKET
u32,u32,u32,u32,u32,u32,u32
0,1,341312,4121690,1,1,1


In [8]:
# casting gas distribution probability to float from string 
A = A.with_columns(
    pl.col("GAS_DISTRIBUTION_PROBABILITY").cast(pl.Float32, strict=False)
)

print(f'new GAS_DISTRIBUTION_PROBABILITY column datatype{A.collect_schema()["GAS_DISTRIBUTION_PROBABILITY"]}')

A.select(
    pl.col("GAS_DISTRIBUTION_PROBABILITY").is_null().sum()
).collect()

new GAS_DISTRIBUTION_PROBABILITY column datatypeFloat32


GAS_DISTRIBUTION_PROBABILITY
u32
4121690


In [9]:
# casting date column to datetime from string
A = A.with_columns(
    pl.col("DATE")
    .str.strptime(
        pl.Date,
        format="%Y%m%d",
        strict=False
    )
    .alias("DATE")
)

date_range = (
    A.select(
        pl.col("DATE").min().alias("first_date"),
        pl.col("DATE").max().alias("last_date"),
        pl.col("DATE").n_unique().alias("unique_dates"),
        pl.col("DATE").null_count().alias("null_dates"),
    )
    .collect()
)

date_range

first_date,last_date,unique_dates,null_dates
date,date,u32,u32
2013-09-13,2015-10-19,727,0


In [10]:
categorical_cols = [
    "CUSTOMER_TYPE",
    "ELECTRICITY_PRODUCT",
    "MARKET",
]

# casting categorical columns to Categorical type
A = A.with_columns(
    pl.col(col).cast(pl.Categorical, strict=False)
    for col in categorical_cols
)

In [11]:
# replace empty strings in string columns (Municipality and Supply registration period) and categorical with nulls
string_cols = ["MUNICIPALITY", "SUPPLY_REGISTRATION_PERIOD"]

A = A.with_columns([
    pl.when(pl.col(c) == "")
      .then(None)
      .otherwise(pl.col(c))
      .alias(c)
    for c in string_cols+categorical_cols
])

## Nulls summary

In [12]:
# Missing values across the complete table
all_cols = A.collect_schema().names()

null_summary = (
    A.select([
        pl.col(c).null_count().alias(c)
        for c in all_cols
    ])
    .collect()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["null_count"]
    )
    .with_columns(
        (pl.col("null_count") / A_rows * 100).alias("null_percent")
    )
    .sort("null_count", descending=True)
)

null_summary

column,null_count,null_percent
str,u32,f64
"""GAS_DISTRIBUTION_PROBABILITY""",4121690,25.512256
"""SUPPLY_REGISTRATION_PERIOD""",341312,2.112638
"""MUNICIPALITY""",1,0.000006
"""CUSTOMER_TYPE""",1,0.000006
"""ELECTRICITY_PRODUCT""",1,0.000006
…,…,…
"""REACTIVE_H24""",0,0.0
"""TIME_H25""",0,0.0
"""ACTIVE_H25""",0,0.0


## Number of Customers and Municipalities

In [13]:
A.select("MUNICIPALITY").unique().sort("MUNICIPALITY").head().collect()

MUNICIPALITY
str
null
"""A CORUÃ‘A"""
"""ABIA DE LAS TORRES"""
"""ABRERA"""
"""ACEBEDO"""


In [14]:
A.filter(
    pl.col("MUNICIPALITY").is_null()
).collect()

DATE,TIME_H1,ACTIVE_H1,REACTIVE_H1,TIME_H2,ACTIVE_H2,REACTIVE_H2,TIME_H3,ACTIVE_H3,REACTIVE_H3,TIME_H4,ACTIVE_H4,REACTIVE_H4,TIME_H5,ACTIVE_H5,REACTIVE_H5,TIME_H6,ACTIVE_H6,REACTIVE_H6,TIME_H7,ACTIVE_H7,REACTIVE_H7,TIME_H8,ACTIVE_H8,REACTIVE_H8,TIME_H9,ACTIVE_H9,REACTIVE_H9,TIME_H10,ACTIVE_H10,REACTIVE_H10,TIME_H11,ACTIVE_H11,REACTIVE_H11,TIME_H12,ACTIVE_H12,REACTIVE_H12,…,TIME_H16,ACTIVE_H16,REACTIVE_H16,TIME_H17,ACTIVE_H17,REACTIVE_H17,TIME_H18,ACTIVE_H18,REACTIVE_H18,TIME_H19,ACTIVE_H19,REACTIVE_H19,TIME_H20,ACTIVE_H20,REACTIVE_H20,TIME_H21,ACTIVE_H21,REACTIVE_H21,TIME_H22,ACTIVE_H22,REACTIVE_H22,TIME_H23,ACTIVE_H23,REACTIVE_H23,TIME_H24,ACTIVE_H24,REACTIVE_H24,TIME_H25,ACTIVE_H25,REACTIVE_H25,MUNICIPALITY,SUPPLY_REGISTRATION_PERIOD,GAS_DISTRIBUTION_PROBABILITY,CUSTOMER_ID,CUSTOMER_TYPE,ELECTRICITY_PRODUCT,MARKET
date,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,str,str,f32,i64,cat,cat,cat
2015-04-06,1,107,0,2,64,0,3,58,0,4,57,0,5,54,0,6,54,1,7,53,0,8,185,12,9,74,9,10,196,4,11,79,9,12,114,3,…,16,117,4,17,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,null,null,null,0,null,null,null


In [15]:
# customer with id 0 has unknown municipality, customer type, electricity product and market and only comprises of one record from the dataset.
A.filter(
    pl.col("CUSTOMER_ID") == 0
).collect()

DATE,TIME_H1,ACTIVE_H1,REACTIVE_H1,TIME_H2,ACTIVE_H2,REACTIVE_H2,TIME_H3,ACTIVE_H3,REACTIVE_H3,TIME_H4,ACTIVE_H4,REACTIVE_H4,TIME_H5,ACTIVE_H5,REACTIVE_H5,TIME_H6,ACTIVE_H6,REACTIVE_H6,TIME_H7,ACTIVE_H7,REACTIVE_H7,TIME_H8,ACTIVE_H8,REACTIVE_H8,TIME_H9,ACTIVE_H9,REACTIVE_H9,TIME_H10,ACTIVE_H10,REACTIVE_H10,TIME_H11,ACTIVE_H11,REACTIVE_H11,TIME_H12,ACTIVE_H12,REACTIVE_H12,…,TIME_H16,ACTIVE_H16,REACTIVE_H16,TIME_H17,ACTIVE_H17,REACTIVE_H17,TIME_H18,ACTIVE_H18,REACTIVE_H18,TIME_H19,ACTIVE_H19,REACTIVE_H19,TIME_H20,ACTIVE_H20,REACTIVE_H20,TIME_H21,ACTIVE_H21,REACTIVE_H21,TIME_H22,ACTIVE_H22,REACTIVE_H22,TIME_H23,ACTIVE_H23,REACTIVE_H23,TIME_H24,ACTIVE_H24,REACTIVE_H24,TIME_H25,ACTIVE_H25,REACTIVE_H25,MUNICIPALITY,SUPPLY_REGISTRATION_PERIOD,GAS_DISTRIBUTION_PROBABILITY,CUSTOMER_ID,CUSTOMER_TYPE,ELECTRICITY_PRODUCT,MARKET
date,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,str,str,f32,i64,cat,cat,cat
2015-04-06,1,107,0,2,64,0,3,58,0,4,57,0,5,54,0,6,54,1,7,53,0,8,185,12,9,74,9,10,196,4,11,79,9,12,114,3,…,16,117,4,17,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,null,null,null,0,null,null,null


In [16]:
# remove customer with CUSTOMER_ID equal to 0 because there's only one record with that customer ID, and their municipality/location is unkown, proving unuseful for further analysis.
A = A.lazy().remove(pl.col("CUSTOMER_ID") == 0)

In [17]:
customer_geo_overview = (
    A.select([
        pl.col("CUSTOMER_ID").n_unique().alias("unique_customers"),
        pl.col("CUSTOMER_ID").null_count().alias("missing_customer_ids"),
        pl.col("MUNICIPALITY").n_unique().alias("unique_municipalities"),
        pl.col("MUNICIPALITY").null_count().alias("missing_municipalities"),
    ])
    .collect()
)

customer_geo_overview

unique_customers,missing_customer_ids,unique_municipalities,missing_municipalities
u32,u32,u32,u32
99999,0,808,0


## Time-position variables removal

There are 25 variables named `TIME_H1` to `TIME_H25`. The documentation associated with the dataset describes the original H1–H25 fields as the hour/time positions within the day. These columns do not add any information and could be dropped to reduce columns.

In [18]:
time_cols = [f"TIME_H{i}" for i in range(1, 26)]

time_unique_counts = (
    A.select([
        pl.col(c).n_unique().alias(c)
        for c in time_cols
    ])
    .collect()
)

time_unique_counts

TIME_H1,TIME_H2,TIME_H3,TIME_H4,TIME_H5,TIME_H6,TIME_H7,TIME_H8,TIME_H9,TIME_H10,TIME_H11,TIME_H12,TIME_H13,TIME_H14,TIME_H15,TIME_H16,TIME_H17,TIME_H18,TIME_H19,TIME_H20,TIME_H21,TIME_H22,TIME_H23,TIME_H24,TIME_H25
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [19]:
A.select(time_cols).head(3).collect()

TIME_H1,TIME_H2,TIME_H3,TIME_H4,TIME_H5,TIME_H6,TIME_H7,TIME_H8,TIME_H9,TIME_H10,TIME_H11,TIME_H12,TIME_H13,TIME_H14,TIME_H15,TIME_H16,TIME_H17,TIME_H18,TIME_H19,TIME_H20,TIME_H21,TIME_H22,TIME_H23,TIME_H24,TIME_H25
i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25
1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25
1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25


In [20]:
A.select([
    pl.col(c).null_count().alias(c)
    for c in time_cols
]).collect()

TIME_H1,TIME_H2,TIME_H3,TIME_H4,TIME_H5,TIME_H6,TIME_H7,TIME_H8,TIME_H9,TIME_H10,TIME_H11,TIME_H12,TIME_H13,TIME_H14,TIME_H15,TIME_H16,TIME_H17,TIME_H18,TIME_H19,TIME_H20,TIME_H21,TIME_H22,TIME_H23,TIME_H24,TIME_H25
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [21]:
# removing time position variables reduces columns from 83 to 58
A = A.drop(time_cols)

## Checking exact duplicates

There are two different concepts:

- an **exact duplicate** has all variables identical; can be easily deduplicated.
- a repeated `CUSTOMER_ID + DATE` may contain different measurements and therefore requires advanced deduplicated methods. In this case we decided to take the row with the max consumption out of its (customer,date) duplicates.

In [22]:
customer_day_counts = (
    A.select(["CUSTOMER_ID", "DATE"])
    .group_by(["CUSTOMER_ID", "DATE"])
    .agg(
        pl.len().alias("records_per_customer_day")
    )
)

customer_day_frequency = (
    customer_day_counts
    .group_by("records_per_customer_day")
    .agg(
        pl.len().alias("customer_days")
    )
    .sort("records_per_customer_day")
    .collect()
)

customer_day_frequency

records_per_customer_day,customer_days
u32,u32
1,15815820
2,167940
3,746
4,444
5,2


In [23]:
active_cols = [f"ACTIVE_H{i}" for i in range(1, 26)]
multi_keys = customer_day_counts.filter(pl.col("records_per_customer_day") > 1).select(["CUSTOMER_ID", "DATE"])

diagnostic = (
    A.join(multi_keys, on=["CUSTOMER_ID", "DATE"], how="inner")
    .with_columns(
        pl.sum_horizontal([pl.col(c).is_not_null() for c in active_cols]).alias("hours_populated")
    )
    .group_by(["CUSTOMER_ID", "DATE"])
    .agg([
        pl.len().alias("n_records"),
        pl.col("hours_populated").sum().alias("hours_populated_summed_across_records"),
        pl.col("hours_populated").max().alias("hours_populated_in_fullest_record"),
        pl.col("CUSTOMER_TYPE").n_unique().alias("n_distinct_customer_types"),
        pl.col("MARKET").n_unique().alias("n_distinct_markets"),
        pl.col("ELECTRICITY_PRODUCT").n_unique().alias("n_distinct_products"),
        
        # Calculates how many individual hours contain >1 unique, non-null values 
        pl.sum_horizontal([
            (pl.col(c).n_unique() > 1)
            for c in active_cols
        ]).alias("n_conflicting_hours")
    ])
    .collect()
)

diagnostic.select([
    (pl.col("n_distinct_customer_types") > 1).sum().alias("groups_with_customer_type_change"),
    (pl.col("n_distinct_markets") > 1).sum().alias("groups_with_market_change"),
    (pl.col("n_distinct_products") > 1).sum().alias("groups_with_product_change"),
    ((pl.col("hours_populated_summed_across_records") >= 20) &
     (pl.col("hours_populated_in_fullest_record") < 20)).sum().alias("groups_looking_complementary"),
    (pl.col("n_conflicting_hours") > 0).sum().alias("groups_with_conflicting_energy"),
])

groups_with_customer_type_change,groups_with_market_change,groups_with_product_change,groups_looking_complementary,groups_with_conflicting_energy
u32,u32,u32,u32,u32
0,418,418,0,127216


In [24]:
identical_energy_keys = (
    diagnostic.lazy()
    .filter(pl.col("n_conflicting_hours") == 0)
    .select(["CUSTOMER_ID", "DATE"])
)

identical_energy_records = (
    A.join(identical_energy_keys, on=["CUSTOMER_ID", "DATE"], how="inner")
    .sort(["CUSTOMER_ID", "DATE"])
    .head(4)  
    .collect()
)
identical_energy_records

DATE,ACTIVE_H1,REACTIVE_H1,ACTIVE_H2,REACTIVE_H2,ACTIVE_H3,REACTIVE_H3,ACTIVE_H4,REACTIVE_H4,ACTIVE_H5,REACTIVE_H5,ACTIVE_H6,REACTIVE_H6,ACTIVE_H7,REACTIVE_H7,ACTIVE_H8,REACTIVE_H8,ACTIVE_H9,REACTIVE_H9,ACTIVE_H10,REACTIVE_H10,ACTIVE_H11,REACTIVE_H11,ACTIVE_H12,REACTIVE_H12,ACTIVE_H13,REACTIVE_H13,ACTIVE_H14,REACTIVE_H14,ACTIVE_H15,REACTIVE_H15,ACTIVE_H16,REACTIVE_H16,ACTIVE_H17,REACTIVE_H17,ACTIVE_H18,REACTIVE_H18,ACTIVE_H19,REACTIVE_H19,ACTIVE_H20,REACTIVE_H20,ACTIVE_H21,REACTIVE_H21,ACTIVE_H22,REACTIVE_H22,ACTIVE_H23,REACTIVE_H23,ACTIVE_H24,REACTIVE_H24,ACTIVE_H25,REACTIVE_H25,MUNICIPALITY,SUPPLY_REGISTRATION_PERIOD,GAS_DISTRIBUTION_PROBABILITY,CUSTOMER_ID,CUSTOMER_TYPE,ELECTRICITY_PRODUCT,MARKET
date,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,str,str,f32,i64,cat,cat,cat
2015-01-25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""A CORUÃ‘A""","""Q22013""",null,2,"""T2""","""P2""","""M1"""
2015-01-25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""A CORUÃ‘A""","""Q22013""",null,2,"""T2""","""P2""","""M1"""
2015-05-21,72,0,47,0,47,0,48,0,47,0,56,0,65,0,108,0,111,0,101,0,74,0,58,0,103,0,90,0,71,0,74,0,67,0,58,0,102,0,73,0,86,0,50,0,53,0,64,0,0,0,"""SANT BOI DE LLOBREGAT""","""Q21975""",0.6,9,"""T2""","""P1""","""M1"""
2015-05-21,72,0,47,0,47,0,48,0,47,0,56,0,65,0,108,0,111,0,101,0,74,0,58,0,103,0,90,0,71,0,74,0,67,0,58,0,102,0,73,0,86,0,50,0,53,0,64,0,0,0,"""SANT BOI DE LLOBREGAT""","""Q21975""",0.6,9,"""T2""","""P1""","""M1"""


In [25]:
rows_to_deduplicate = (
    A.join(identical_energy_keys, on=["CUSTOMER_ID", "DATE"], how="inner")
    .select(pl.len())
    .collect()
    .item()
)
print(f"Raw duplicated rows with consistent energy: {rows_to_deduplicate}")

Raw duplicated rows with consistent energy: 84717


In [26]:
groups_to_collapse = identical_energy_keys.with_columns(
    pl.lit(True).alias("_collapse")
)

# Keep only one record per collapsing group.
# We use an in-group row index so that "first" is well defined and
# every other row of A is left exactly as it was.
A = (
    A
    .with_columns(
        pl.int_range(pl.len())
          .over(["CUSTOMER_ID", "DATE"])
          .alias("_row_in_group")
    )
    .join(
        groups_to_collapse,
        on=["CUSTOMER_ID", "DATE"],
        how="left"
    )
    .filter(
        pl.col("_collapse").is_null()
        | (pl.col("_row_in_group") == 0)
    )
    .drop(["_row_in_group", "_collapse"])
)

In [27]:
n_collapsing_groups = (
    identical_energy_keys.select(pl.len()).collect().item()
)
rows_removed = rows_to_deduplicate - n_collapsing_groups

print(f"Rows removed:             {rows_removed:,}")
print(f"Rows remaining (lazy):    "
      f"{A.select(pl.len()).collect().item():,}")

# Sanity check: no collapsing group should still contain >1 row.
remaining_dupes = (
    A
    .filter(pl.col("CUSTOMER_ID").is_not_null())
    .group_by(["CUSTOMER_ID", "DATE"])
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .select(pl.len())
    .collect()
    .item()
)
print(f"Groups still with >1 row: {remaining_dupes:,}")

Rows removed:             42,801
Rows remaining (lazy):    16,112,923
Groups still with >1 row: 127,216


In [28]:
active_cols = [f"ACTIVE_H{i}" for i in range(1, 26)]

A = (
    A
    .with_columns(
        pl.sum_horizontal([pl.col(c) for c in active_cols]).alias("_temp_total")
    )
    .sort("_temp_total", descending=True)
    .group_by(["CUSTOMER_ID", "DATE"], maintain_order=True)
    .first()
    .drop("_temp_total")
)

remaining = (
    A
    .group_by(["CUSTOMER_ID", "DATE"])
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .select(pl.len())
    .collect()
    .item()
)
print(f"Groups still duplicated: {remaining}")

Groups still duplicated: 0


In [29]:
# for faster analysis runtime after deduplication
# One-time cost
A.sink_parquet(
    PROJECT_ROOT / "data" / "A_cleaned.parquet",
    compression="zstd",
    row_group_size=500_000,
)

# All subsequent work reads from the file — fast because it's columnar
A = pl.scan_parquet(PROJECT_ROOT / "data" / "A_cleaned.parquet")